# Response generation

[Fine-tuning Llama 2 with LoRA](https://deci.ai/blog/fine-tune-llama-2-with-lora-for-question-answering/)

# Fine-tuning
new_model will be the name of your fine-tuned model (saved)

In [1]:
import os, torch, logging
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, HfArgumentParser, TrainingArguments, pipeline
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

from flash_attn import flash_attn_func
import accelerate
# from accelerate import DataLoaderConfiguration
import wandb

In [2]:
from huggingface_hub import login

login(token=os.environ.get("HF_TOKEN", ""), add_to_git_credential=True)
wandb.login(key=os.environ.get("WANDB_API_KEY", ""), relogin=True)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


Token is valid (permission: write).
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
Token has not been saved to git credential helper.
Your token has been saved to /home/user/.cache/huggingface/token
Login successful


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/user/.netrc


True

In [3]:
# Initialize Wandb
wandb_config = {
    "base_model": "llama-2-chat-7b",
    # "tokenizer": arguments.tokenizer,
    # "name_or_path_for_fine_tuned_model": arguments.name_or_path_for_fine_tuned_model,
    # "system_prompt": arguments.system_prompt,
    # "chat_template": chat_template["chat"],
    # "instruction_template": chat_template["instruction"],
    # "response_template": chat_template["response"]
}
wandb.init(
    job_type="fine-tuning",
    config=wandb_config,
    project="emotion-chat-bot-ncu",
    group="candidate_generation",
    # notes=arguments.experiment_detail,
    mode="online",
    resume="auto"
)

wandb: Currently logged in as: yangyx30678. Use `wandb login --relogin` to force relogin


In [4]:
# Dataset
data_name = "mlabonne/guanaco-llama2-1k"
training_data = load_dataset(data_name, split="train")

# Model and tokenizer names
base_model_name = "meta-llama/Llama-2-7b-chat-hf"
refined_model = "llama-2-7b-testing"

# Tokenizer
llama_tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
llama_tokenizer.pad_token = llama_tokenizer.eos_token
llama_tokenizer.padding_side = "right"  # Fix for fp16

# Quantization Config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False
)
wandb.config["quantization_configuration"] = quant_config.to_dict() if quant_config is not None else {}

# Model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    attn_implementation="flash_attention_2",
    quantization_config=quant_config,
    device_map={"": 0},
    use_cache=False,
    low_cpu_mem_usage=True
)

base_model.config.use_cache = False
base_model.config.pretraining_tp = 1

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
# LoRA Config
peft_parameters = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=8,
    bias="none",
    task_type="CAUSAL_LM"
)
wandb.config["lora_configuration"] = peft_parameters.to_dict()

# Training Params
train_params = TrainingArguments(
    output_dir="./results_modified",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=25,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
    report_to=["wandb"],
    overwrite_output_dir=True
    # torch_compile=True
)
wandb.config["trainer_arguments"] = train_params.to_dict()

# Trainer
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=training_data,
    peft_config=peft_parameters,
    dataset_text_field="text",
    tokenizer=llama_tokenizer,
    args=train_params
)

# Training
fine_tuning.train()
wandb.finish()
# Save Model
fine_tuning.model.save_pretrained(refined_model)

/home/user/.cache/pypoetry/virtualenvs/chat-bot-20tW9agt-py3.11/lib/python3.11/site-packages/trl/trainer/sft_trainer.py:225: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(
/home/user/.cache/pypoetry/virtualenvs/chat-bot-20tW9agt-py3.11/lib/python3.11/site-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
The input hidden states seems to be silently casted in float32, this might be related to the fact you have upcasted embedding or layer norm layers in float32. We will cast back the input in torch.float

Step,Training Loss
25,1.343600
50,1.632100
75,1.219900
100,1.443400
125,1.184700
150,1.370500
175,1.177500
200,1.460800
225,1.154200
250,1.521300


Checkpoint destination directory ./results_modified/checkpoint-25 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./results_modified/checkpoint-50 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./results_modified/checkpoint-75 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./results_modified/checkpoint-100 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./results_modified/checkpoint-125 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./results_modified/checkpoint-150 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./results_modified/checkpoint-175 already exis

train/epoch,▁▂▃▃▄▅▆▆▇██
train/global_step,▁▂▃▃▄▅▆▆▇██
train/grad_norm,▁▅▁█▁▂▁▄▁▃
train/learning_rate,▁▁▁▁▁▁▁▁▁▁
train/loss,▄█▂▅▁▄▁▅▁▆
train/total_flos,▁
train/train_loss,▁
train/train_runtime,▁
train/train_samples_per_second,▁
train/train_steps_per_second,▁
train/epoch,1.0


In [6]:
inference_model = AutoModelForCausalLM.from_pretrained(
    refined_model, 
    load_in_4bit=True,
    attn_implementation="flash_attention_2",
)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
`low_cpu_mem_usage` was None, now set to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
# Generate Text
query = "How do I use the OpenAI API?"
text_gen = pipeline(task="text-generation", model=inference_model, tokenizer=llama_tokenizer, max_length=200)
output = text_gen(f"<s>[INST] {query} [/INST]")
print(output[0]['generated_text'])

/home/user/.cache/pypoetry/virtualenvs/chat-bot-20tW9agt-py3.11/lib/python3.11/site-packages/bitsandbytes/nn/modules.py:226: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(f'Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.')


<s>[INST] How do I use the OpenAI API? [/INST] The OpenAI API is a web API that allows you to use OpenAI models and capabilities through a simple, consistent interface. Here are the steps to use the OpenAI API:

1. Sign up for an OpenAI account: You will need to create an OpenAI account to use the OpenAI API. You can sign up for a free account on the OpenAI website.

2. Obtain an API token: Once you have created an OpenAI account, you can obtain an API token by logging into your OpenAI account and following the instructions provided by OpenAI.

3. Use the OpenAI API: Once you have obtained an API token, you can use the OpenAI API to make requests to OpenAI models. The API allows you to perform a variety of tasks, including:

* Making requests to OpenAI models
* Uploading data to OpenAI
